# UrbanHub - Jumeau Numérique Urbain
### Plateforme Big Data & Tableau de Bord Smart City (Restitution Gold)

> **Architecture :** Data Lake Medallion (Couche `Gold` Cloudflare R2 / MinIO)  
> **Flux Ingestion :** Météo Batch (Open-Meteo) | Mobilité Streaming (CityBikes) | Pollution IoT (OpenAQ v3)

In [ ]:
import os
import io
import json
import itertools
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
try:
    import folium
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False
import ipywidgets as widgets
from IPython.display import display, clear_output
from minio import Minio

# Icônes SVG inline (remplacent les emojis) : [[ico:nom]] est résolu dans tout HTML() affiché
import re as _re
from IPython.display import HTML as _HTML
_ICONS = {'alert': ('<path d="m21.73 18-8-14a2 2 0 0 0-3.48 0l-8 14A2 2 0 0 0 4 21h16a2 2 0 0 0 1.73-3"/><path d="M12 9v4"/><path d="M12 17h.01"/>', '#d97706'), 'bike': ('<circle cx="18.5" cy="17.5" r="3.5"/><circle cx="5.5" cy="17.5" r="3.5"/><circle cx="15" cy="5" r="1"/><path d="M12 17.5V14l-3-3 4-3 2 3h2"/>', '#16a34a'), 'check-circle': ('<circle cx="12" cy="12" r="10"/><path d="m9 12 2 2 4-4"/>', '#16a34a'), 'clock': ('<circle cx="12" cy="12" r="10"/><path d="M12 6v6l4 2"/>', '#64748b'), 'dot-green': ('<circle cx="12" cy="12" r="7" fill="#16a34a"/>', '#16a34a'), 'info': ('<circle cx="12" cy="12" r="10"/><path d="M12 16v-4"/><path d="M12 8h.01"/>', '#2563eb'), 'map': ('<path d="M14.106 5.553a2 2 0 0 0 1.788 0l3.659-1.83A1 1 0 0 1 21 4.619v12.764a1 1 0 0 1-.553.894l-4.553 2.277a2 2 0 0 1-1.788 0l-4.212-2.106a2 2 0 0 0-1.788 0l-3.659 1.83A1 1 0 0 1 3 19.381V6.618a1 1 0 0 1 .553-.894l4.553-2.277a2 2 0 0 1 1.788 0z"/><path d="M15 5.764v15"/><path d="M9 3.236v15"/>', '#16a34a'), 'thermometer': ('<path d="M14 4v10.54a4 4 0 1 1-4 0V4a2 2 0 0 1 4 0Z"/>', '#0ea5e9'), 'wind': ('<path d="M17.7 7.7a2.5 2.5 0 1 1 1.8 4.3H2"/><path d="M9.6 4.6A2 2 0 1 1 11 8H2"/><path d="M12.6 19.4A2 2 0 1 0 14 16H2"/>', '#8b5cf6'), 'x-circle': ('<circle cx="12" cy="12" r="10"/><path d="m15 9-6 6"/><path d="m9 9 6 6"/>', '#dc2626')}
def _svg(m):
    body, color = _ICONS.get(m.group(1), ('', '#64748b'))
    return ("<svg xmlns='http://www.w3.org/2000/svg' width='16' height='16' viewBox='0 0 24 24' fill='none' "
            f"stroke='{color}' stroke-width='2' stroke-linecap='round' stroke-linejoin='round' "
            f"style='vertical-align:-3px'>{body}</svg>").replace('"', "'")
def HTML(s):
    return _HTML(_re.sub(r'\[\[ico:([\w-]+)\]\]', _svg, s))

# 1. Chargement robuste du .env
def load_env():
    candidates = ["../.env", ".env", "/app/.env"]
    for p in candidates:
        if os.path.exists(p):
            with open(p, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, v = line.split("=", 1)
                        os.environ[k.strip()] = v.strip().strip("'\"")
            return p
    return None

load_env()

# 2. Chargement du référentiel des villes
config_path = "../config/cities.yml" if os.path.exists("../config/cities.yml") else "config/cities.yml"
with open(config_path, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

cities_meta = {c["id"]: c for c in config.get("cities", [])}
cities_dict = {c["name"]: c["id"] for c in config.get("cities", [])}

# 3. Connexion S3 (MinIO local ou Cloudflare R2)
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "localhost:9000")
SECURE = os.getenv("MINIO_SECURE", "false").lower() == "true"
BUCKET_GOLD = os.getenv("MINIO_BUCKET_GOLD", "urbanhub-gold")

# Délais courts : sans eux, chaque lecture d'un lac injoignable attend environ 25 s (nouvelles tentatives de minio-py)
# et la page reste figée plusieurs minutes sans message.
import urllib3
import certifi
_http = urllib3.PoolManager(
    timeout=urllib3.Timeout(connect=3, read=30),
    retries=urllib3.Retry(total=1, backoff_factor=0.2, status_forcelist=[500, 502, 503, 504]),
    cert_reqs="CERT_REQUIRED",
    ca_certs=certifi.where(),
)
client = Minio(
    MINIO_ENDPOINT,
    access_key=os.getenv("MINIO_ACCESS_KEY"),
    secret_key=os.getenv("MINIO_SECRET_KEY"),
    secure=SECURE,
    http_client=_http,
)

def read_parquet_table(keys, columns=None):
    if isinstance(keys, str):
        keys = [keys]
    for k in keys:
        try:
            resp = client.get_object(BUCKET_GOLD, k)
            df = pd.read_parquet(io.BytesIO(resp.read()), columns=columns)
            resp.close()
            resp.release_conn()
            return df
        except Exception:
            continue
    return None

def read_json_meta(key):
    try:
        resp = client.get_object(BUCKET_GOLD, key)
        data = json.loads(resp.read().decode('utf-8'))
        resp.close()
        resp.release_conn()
        return data
    except Exception:
        return None

# Ville transmise par l'adresse (?city=lyon) : conservée lors du rechargement automatique de la page
from urllib.parse import parse_qs, urlparse
VILLE_URL = parse_qs(urlparse(os.getenv("VOILA_REQUEST_URL", "")).query).get("city", [None])[0]

# Fraîcheur des données : dernière donnée de chaque source, lue directement dans Gold
from datetime import datetime, timezone

DASHBOARD_REFRESH_SECONDS = int(os.getenv("DASHBOARD_REFRESH_SECONDS", "300"))
# Âge maximal (heures) au-delà duquel une source est signalée en retard
SEUILS_FRAICHEUR_H = {"Météo": 3, "Mobilité": 0.5, "Pollution": 8, "Croisement": 8}

def _derniere_heure(df, col):
    if df is None or df.empty or col not in df:
        return None
    ts = pd.to_datetime(df[col], utc=True, errors="coerce").max()
    return None if pd.isna(ts) else ts

def _age_texte(age_h):
    if age_h < 1:
        return f"{int(age_h * 60)} min"
    if age_h < 48:
        return f"{age_h:.1f} h"
    return f"{age_h / 24:.0f} j"

def fraicheur_html():
    """Tableau de fraîcheur : dernière donnée, âge et état de chaque source."""
    maintenant = pd.Timestamp.now(tz="UTC")
    intervalle_texte = (f"{DASHBOARD_REFRESH_SECONDS} s" if DASHBOARD_REFRESH_SECONDS < 120
                        else f"{DASHBOARD_REFRESH_SECONDS // 60} min")
    diag = read_json_meta("kpis/diagnostic_couverture_croisee.json") or {}
    croisement = pd.to_datetime(diag.get("date_diagnostic_utc"), utc=True, errors="coerce")
    sources = [
        ("Météo (Open-Meteo)", _derniere_heure(read_parquet_table(["kpis/weather_hourly.parquet"], columns=["hour_utc"]), "hour_utc"), SEUILS_FRAICHEUR_H["Météo"]),
        ("Mobilité (CityBikes)", _derniere_heure(read_parquet_table(["citybikes/kpis/mobility_station_latest.parquet", "kpis/mobility_station_latest.parquet"], columns=["observed_at_utc"]), "observed_at_utc"), SEUILS_FRAICHEUR_H["Mobilité"]),
        ("Pollution (OpenAQ)", _derniere_heure(read_parquet_table(["kpis/pollution_hourly.parquet"], columns=["hour_utc"]), "hour_utc"), SEUILS_FRAICHEUR_H["Pollution"]),
        ("Croisement (city_hourly)", None if pd.isna(croisement) else croisement, SEUILS_FRAICHEUR_H["Croisement"]),
    ]
    lignes = ""
    for nom, derniere, seuil in sources:
        if derniere is None:
            etat = "<span style='color:#7f8c8d;'>[[ico:x-circle]] Indisponible</span>"
            quand, age = "-", "-"
        else:
            age_h = max(0.0, (maintenant - derniere).total_seconds() / 3600)
            ok = age_h <= seuil
            etat = ("<span style='color:#27ae60;font-weight:bold;'>[[ico:check-circle]] À jour</span>" if ok
                    else "<span style='color:#e67e22;font-weight:bold;'>[[ico:alert]] En retard</span>")
            quand, age = derniere.strftime("%d/%m/%Y %H:%M"), _age_texte(age_h)
        lignes += (f"<tr><td style='padding:6px 12px;text-align:left;white-space:nowrap;'>{nom}</td><td style='padding:6px 12px;text-align:left;white-space:nowrap;'>{quand} UTC</td>"
                   f"<td style='padding:6px 12px;text-align:left;white-space:nowrap;'>{age}</td><td style='padding:6px 12px;text-align:left;white-space:nowrap;'>{etat}</td></tr>")
    return f"""
    <div style='background:#f8f9fa;border-radius:8px;padding:12px 16px;margin-bottom:18px;'>
        <div style='font-weight:bold;color:#2c3e50;margin-bottom:6px;'>[[ico:clock]] Fraîcheur des données</div>
        <table style='border-collapse:collapse;font-size:13px;width:100%;'>
            <tr style='color:#7f8c8d;text-align:left;'><th style='padding:4px 12px;text-align:left;white-space:nowrap;'>Source</th><th style='padding:4px 12px;text-align:left;white-space:nowrap;'>Dernière donnée</th><th style='padding:4px 12px;text-align:left;white-space:nowrap;'>Âge</th><th style='padding:4px 12px;text-align:left;white-space:nowrap;'>État</th></tr>
            {lignes}
        </table>
        <div style='color:#95a5a6;font-size:12px;margin-top:6px;'>Actualisé à {maintenant:%H:%M:%S} UTC. Rafraîchissement automatique toutes les {intervalle_texte}.</div>
    </div>
    """

def script_rechargement(ville_id):
    """Script qui recharge la page à intervalle fixe en gardant la ville dans l'adresse.

    Le minuteur n'est armé qu'une fois ; chaque affichage met seulement à jour la ville retenue."""
    return ("<script>window.__urbanhub_ville = '" + str(ville_id) + "';"
            "if (!window.__urbanhub_minuteur) { window.__urbanhub_minuteur = setTimeout(function () {"
            "var u = new URL(window.location.href); u.searchParams.set('city', window.__urbanhub_ville);"
            "window.location.href = u.toString(); }, " + str(DASHBOARD_REFRESH_SECONDS * 1000) + "); }</script>")

def verifier_lac():
    """Teste l'accès au lac une seule fois, au démarrage, avec un message clair en cas d'échec."""
    conseil = ("Vérifier MINIO_ENDPOINT, MINIO_ACCESS_KEY, MINIO_SECRET_KEY et MINIO_SECURE dans le fichier .env "
               "(modèle : .env.example), puis relancer : docker compose restart dashboard.")
    try:
        if client.bucket_exists(BUCKET_GOLD):
            return True, f"[OK] Data Lake connecté (bucket Gold : {BUCKET_GOLD})"
        return False, (f"[ERREUR] Le bucket {BUCKET_GOLD} est introuvable : le .env pointe peut-être vers un MinIO local vide "
                       f"au lieu de Cloudflare R2. {conseil}")
    except Exception as erreur:
        return False, f"[ERREUR] Data Lake injoignable ({type(erreur).__name__}). {conseil}"


LAC_OK, LAC_MESSAGE = verifier_lac()
print(LAC_MESSAGE)

## Sélecteurs de Supervision Urbaine

In [ ]:
city_selector = widgets.Dropdown(
    options=list(cities_dict.keys()),
    value=next((nom for nom, cid in cities_dict.items() if cid == VILLE_URL), list(cities_dict.keys())[0] if cities_dict else None),
    description="Ville :",
    layout=widgets.Layout(width="320px")
)

refresh_btn = widgets.Button(
    description="Actualiser les KPI",
    button_style="primary",
    icon="sync",
    layout=widgets.Layout(width="200px")
)

output_dashboard = widgets.Output()

controls_box = widgets.HBox(
    [city_selector, refresh_btn],
    layout=widgets.Layout(margin="10px 0 20px 0", align_items="center")
)
display(controls_box)
display(output_dashboard)

## Restitution & Visualisations Multidimensionnelles

In [ ]:
_cache_analyses = {}
_cache_tables = {}
_cache_nb = {}


def lire_historique(cle, keys, columns=None):
    # Les tables d'historique ne changent pas pendant la session : une seule lecture par noyau
    if cle not in _cache_tables:
        _cache_tables[cle] = read_parquet_table(keys, columns=columns)
    return _cache_tables[cle]


def render_dashboard(b=None):
    selected_name = city_selector.value
    selected_id = cities_dict.get(selected_name, "paris")
    city_info = cities_meta.get(selected_id, {"latitude": 48.8566, "longitude": 2.3522})
    
    with output_dashboard:
        clear_output(wait=True)
        if not LAC_OK:
            display(HTML("<div style='background:#fdecea;border-left:5px solid #dc2626;padding:16px 20px;border-radius:8px;'>"
                         "<h4 style='margin:0 0 8px 0;color:#b91c1c;'>[[ico:x-circle]] Le Data Lake est inaccessible</h4>"
                         f"<p style='margin:0;font-size:13px;color:#7f1d1d;'>{LAC_MESSAGE}</p></div>"))
            return
        
        # En-tête dynamique
        header_html = f"""
        <div style='background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); color: white; padding: 18px 25px; border-radius: 10px; margin-bottom: 20px; box-shadow: 0 4px 6px rgba(0,0,0,0.1);'>
            <div style='display: flex; justify-content: space-between; align-items: center;'>
                <div>
                    <h2 style='margin: 0; font-size: 24px;'>Supervision Métropolitaine : {selected_name}</h2>
                    <p style='margin: 5px 0 0 0; opacity: 0.85; font-size: 14px;'>Identifiant : <code>{selected_id}</code> | Coordonnées : {city_info.get('latitude', 0):.4f}°N, {city_info.get('longitude', 0):.4f}°E</p>
                </div>
                <div style='text-align: right;'>
                    <span style='background: rgba(255,255,255,0.2); padding: 6px 14px; border-radius: 20px; font-weight: bold; font-size: 13px;'>[[ico:dot-green]] Data Lake Connecté</span>
                </div>
            </div>
        </div>
        """
        display(HTML(header_html))
        display(HTML(fraicheur_html()))
        display(HTML(script_rechargement(selected_id)))
        
        # Chargement des tables
        df_w = read_parquet_table(["kpis/weather_daily.parquet"])
        df_m = read_parquet_table(["citybikes/kpis/mobility_station_latest.parquet", "kpis/mobility_station_latest.parquet"])
        df_m_hourly = read_parquet_table(["citybikes/kpis/mobility_hourly.parquet", "kpis/mobility_hourly.parquet"])
        df_p = read_parquet_table(["kpis/pollution_hourly.parquet"])
        p_summary = read_json_meta("kpis/pollution_summary.json")
        df_c = read_parquet_table(["kpis/city_hourly.parquet"])
        
        # Création des onglets
        tab = widgets.Tab()
        tab_titles = ["Météo", "Mobilité", "Qualité de l'air", "Croisement", "Historique", "Analyses 5 ans"]
        out_w, out_m, out_p, out_c, out_h, out_a = [widgets.Output() for _ in range(6)]
        tab.children = [out_w, out_m, out_p, out_c, out_h, out_a]
        for i, titre in enumerate(tab_titles):
            tab.set_title(i, titre)
        display(tab)  # les onglets apparaissent tout de suite et se remplissent ensuite
        
        # ==========================================================
        # ONGLET 1 : MÉTÉO
        # ==========================================================
        with out_w:
            if df_w is not None and not df_w.empty:
                sub_w = df_w[df_w["city_id"] == selected_id].copy()
                if not sub_w.empty:
                    t_mean = sub_w["temperature_mean"].mean()
                    t_min = sub_w["temperature_min"].min()
                    t_max = sub_w["temperature_max"].max()
                    w_max = sub_w["wind_speed_max"].max()
                    nb_days = len(sub_w)
                    
                    kpi_w_html = f"""
                    <div style='display: grid; grid-template-columns: repeat(4, 1fr); gap: 15px; margin-bottom: 20px;'>
                        <div style='background: #f8f9fa; border-left: 5px solid #3498db; padding: 15px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 13px; font-weight: bold;'>TEMPÉRATURE MOYENNE</div>
                            <div style='font-size: 26px; font-weight: bold; color: #2c3e50; margin-top: 5px;'>{t_mean:.1f} °C</div>
                            <div style='color: #95a5a6; font-size: 12px; margin-top: 3px;'>Sur {nb_days} jours observés</div>
                        </div>
                        <div style='background: #f8f9fa; border-left: 5px solid #2ecc71; padding: 15px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 13px; font-weight: bold;'>EXTRÊME MINIMAL</div>
                            <div style='font-size: 26px; font-weight: bold; color: #27ae60; margin-top: 5px;'>{t_min:.1f} °C</div>
                            <div style='color: #95a5a6; font-size: 12px; margin-top: 3px;'>Record froid observé</div>
                        </div>
                        <div style='background: #f8f9fa; border-left: 5px solid #e74c3c; padding: 15px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 13px; font-weight: bold;'>EXTRÊME MAXIMAL</div>
                            <div style='font-size: 26px; font-weight: bold; color: #c0392b; margin-top: 5px;'>{t_max:.1f} °C</div>
                            <div style='color: #95a5a6; font-size: 12px; margin-top: 3px;'>Record chaleur observé</div>
                        </div>
                        <div style='background: #f8f9fa; border-left: 5px solid #f39c12; padding: 15px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 13px; font-weight: bold;'>VENT MAXIMAL</div>
                            <div style='font-size: 26px; font-weight: bold; color: #d35400; margin-top: 5px;'>{w_max:.1f} m/s</div>
                            <div style='color: #95a5a6; font-size: 12px; margin-top: 3px;'>Vent horaire maximal à 10 m</div>
                        </div>
                    </div>
                    """
                    display(HTML(kpi_w_html))
                    
                    # Graphique d'évolution temporelle
                    fig, ax = plt.subplots(figsize=(11, 4), dpi=100)
                    sub_w = sub_w.sort_values("day_utc").tail(180) # Derniers 6 mois pour lisibilité
                    ax.plot(pd.to_datetime(sub_w["day_utc"]), sub_w["temperature_mean"], label="Température Moyenne (°C)", color="#2980b9", lw=2)
                    ax.fill_between(pd.to_datetime(sub_w["day_utc"]), sub_w["temperature_min"], sub_w["temperature_max"], color="#3498db", alpha=0.2, label="Amplitude Min-Max")
                    ax.set_title(f"Chronologie des Températures Journalières - {selected_name} (Derniers 180 jours)", fontsize=13, fontweight="bold", pad=10)
                    ax.set_xlabel("Date")
                    ax.set_ylabel("Température (°C)")
                    ax.grid(True, linestyle="--", alpha=0.5)
                    ax.legend(loc="upper left")
                    plt.tight_layout()
                    plt.show()
                else:
                    display(HTML(f"<p style='color:#e67e22;'>[[ico:alert]] Aucune donnée météo trouvée pour <code>{selected_id}</code>.</p>"))
            else:
                display(HTML("<p style='color:#e74c3c;'>[[ico:x-circle]] Table Gold météo introuvable.</p>"))
        
        # ==========================================================
        # ONGLET 2 : MOBILITÉ CITYBIKES
        # ==========================================================
        with out_m:
            if df_m is not None and not df_m.empty:
                sub_m = df_m[df_m["city_id"] == selected_id].copy()
                if not sub_m.empty:
                    tot_stations = len(sub_m)
                    tot_bikes = int(sub_m["bikes_available"].sum())
                    tot_slots = int(sub_m["free_slots"].sum())
                    nb_empty = int((sub_m["bikes_available"] == 0).sum())
                    nb_full = int((sub_m["free_slots"] == 0).sum())
                    pct_dispo = (tot_bikes / (tot_bikes + tot_slots) * 100) if (tot_bikes + tot_slots) > 0 else 0
                    
                    kpi_m_html = f"""
                    <div style='display: grid; grid-template-columns: repeat(4, 1fr); gap: 15px; margin-bottom: 20px;'>
                        <div style='background: #f8f9fa; border-left: 5px solid #27ae60; padding: 15px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 13px; font-weight: bold;'>TOTAL STATIONS</div>
                            <div style='font-size: 26px; font-weight: bold; color: #2c3e50; margin-top: 5px;'>{tot_stations}</div>
                            <div style='color: #27ae60; font-size: 12px; margin-top: 3px;'>Stations du dernier relevé</div>
                        </div>
                        <div style='background: #f8f9fa; border-left: 5px solid #2980b9; padding: 15px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 13px; font-weight: bold;'>VÉLOS DISPONIBLES</div>
                            <div style='font-size: 26px; font-weight: bold; color: #2980b9; margin-top: 5px;'>{tot_bikes}</div>
                            <div style='color: #95a5a6; font-size: 12px; margin-top: 3px;'>{pct_dispo:.1f}% de taux de remplissage</div>
                        </div>
                        <div style='background: #f8f9fa; border-left: 5px solid #8e44ad; padding: 15px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 13px; font-weight: bold;'>BORNES LIBRES</div>
                            <div style='font-size: 26px; font-weight: bold; color: #8e44ad; margin-top: 5px;'>{tot_slots}</div>
                            <div style='color: #95a5a6; font-size: 12px; margin-top: 3px;'>Capacité d'accueil retour</div>
                        </div>
                        <div style='background: #f8f9fa; border-left: 5px solid #e74c3c; padding: 15px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 13px; font-weight: bold;'>STATIONS EN RUPTURE</div>
                            <div style='font-size: 26px; font-weight: bold; color: #c0392b; margin-top: 5px;'>{nb_empty}</div>
                            <div style='color: #e67e22; font-size: 12px; margin-top: 3px;'>{nb_full} stations saturées</div>
                        </div>
                    </div>
                    """
                    display(HTML(kpi_m_html))
                    
                    # Carte interactive Folium (OpenStreetMap 100% libre et sans clé API)
                    if HAS_FOLIUM:
                        display(HTML("<h4>[[ico:map]] Cartographie des stations et niveaux de disponibilité (dernier relevé)</h4>"))
                        m = folium.Map(
                            location=[city_info.get("latitude", 48.8566), city_info.get("longitude", 2.3522)],
                            zoom_start=13,
                            tiles="OpenStreetMap"
                        )
                        
                        # Ajout des marqueurs de stations (échantillonnage de 400 stations pour fluidité)
                        sub_sample = sub_m.head(400) if len(sub_m) > 400 else sub_m
                        for _, row in sub_sample.iterrows():
                            lat = row.get("latitude")
                            lon = row.get("longitude")
                            if pd.notna(lat) and pd.notna(lon):
                                bikes = int(row.get("bikes_available", 0))
                                slots = int(row.get("free_slots", 0))
                                name = row.get("station_name", "Station")
                                
                                if bikes == 0:
                                    color = "#e74c3c" # Rouge (rupture)
                                elif bikes < 4:
                                    color = "#f39c12" # Orange (tension)
                                else:
                                    color = "#27ae60" # Vert (dispo)
                                
                                folium.CircleMarker(
                                    location=[lat, lon],
                                    radius=5,
                                    color=color,
                                    fill=True,
                                    fill_color=color,
                                    fill_opacity=0.8,
                                    popup=f"<b>{name}</b><br>Vélos dispo : {bikes}<br>Places libres : {slots}"
                                ).add_to(m)
                        
                        display(HTML(m._repr_html_()))
                    else:
                        display(HTML("<p style='color:#7f8c8d;'><i>Module folium non disponible pour la carte. Aperçu tabulaire :</i></p>"))
                        display(sub_m[['station_name', 'bikes_available', 'free_slots']].head(15))
                else:
                    display(HTML(f"<p style='color:#e67e22;'>[[ico:alert]] Aucune station trouvée pour <code>{selected_id}</code>.</p>"))
            else:
                display(HTML("<p style='color:#e74c3c;'>[[ico:x-circle]] Table Gold mobilité introuvable.</p>"))
        
        # ==========================================================
        # ONGLET 3 : POLLUTION OPENAQ
        # ==========================================================
        with out_p:
            if df_p is not None and not df_p.empty:
                sub_p = df_p[df_p["city_id"] == selected_id].copy()
                if not sub_p.empty:
                    who_thresholds = {"pm25": 15.0, "no2": 25.0, "pm10": 45.0, "o3": 100.0, "co": 4000.0}
                    stats = sub_p.groupby("pollutant")["value_mean"].mean().to_dict()
                    
                    kpi_p_cards = ""
                    for pol, val in stats.items():
                        thr = who_thresholds.get(pol, 50.0)
                        is_danger = val > thr
                        badge = "<span style='color:#e74c3c; font-weight:bold;'>[[ico:alert]] Dépassement OMS</span>" if is_danger else "<span style='color:#27ae60; font-weight:bold;'>[[ico:check-circle]] Conforme OMS</span>"
                        border_c = "#e74c3c" if is_danger else "#27ae60"
                        kpi_p_cards += f"""
                        <div style='background: #f8f9fa; border-left: 5px solid {border_c}; padding: 12px; border-radius: 6px;'>
                            <div style='color: #7f8c8d; font-size: 12px; font-weight: bold;'>POLLUANT : {pol.upper()}</div>
                            <div style='font-size: 22px; font-weight: bold; color: #2c3e50; margin-top: 4px;'>{val:.2f} µg/m³</div>
                            <div style='font-size: 11px; margin-top: 3px;'>Seuil OMS : {thr} µg/m³<br>{badge}</div>
                        </div>
                        """
                    
                    display(HTML(f"<div style='display: grid; grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); gap: 12px; margin-bottom: 20px;'>{kpi_p_cards}</div>"))
                    
                    # Graphique barres comparatif
                    fig, ax = plt.subplots(figsize=(9, 4), dpi=100)
                    pols = [p for p in stats.keys() if p != "co"] # Exclusion CO car échelle différente (mg)
                    vals = [stats[p] for p in pols]
                    thrs = [who_thresholds.get(p, 50) for p in pols]
                    
                    x = np.arange(len(pols))
                    width = 0.35
                    ax.bar(x - width/2, vals, width, label="Moyenne Observée", color="#e67e22")
                    ax.bar(x + width/2, thrs, width, label="Seuil Recommandé OMS", color="#95a5a6", alpha=0.6)
                    ax.set_title(f"Conformité Qualité de l'Air vs Seuils OMS - {selected_name}", fontsize=12, fontweight="bold")
                    ax.set_xticks(x)
                    ax.set_xticklabels([p.upper() for p in pols])
                    ax.set_ylabel("Concentration (µg/m³)")
                    ax.legend()
                    ax.grid(True, linestyle="--", alpha=0.4)
                    plt.tight_layout()
                    plt.show()
                else:
                    display(HTML(f"<p style='color:#e67e22;'>[[ico:alert]] Aucune mesure de pollution trouvée pour <code>{selected_id}</code>.</p>"))
            else:
                display(HTML("<p style='color:#e74c3c;'>[[ico:x-circle]] Table Gold pollution introuvable.</p>"))
        
        # ==========================================================
        # ONGLET 4 : ANALYSE CROISÉE & JUMEAU NUMÉRIQUE
        # ==========================================================
        with out_c:
            if df_c is not None and not df_c.empty:
                sub_c = df_c[df_c["city_id"] == selected_id].copy()
                if not sub_c.empty:
                    nb_tot = len(sub_c)
                    nb_poll = int(sub_c["pollution_no2_ugm3"].notna().sum()) if "pollution_no2_ugm3" in sub_c else 0
                    nb_mob = int(sub_c["mobility_bikes_available"].notna().sum()) if "mobility_bikes_available" in sub_c else 0
                    nb_met = int(sub_c["weather_temperature_c"].notna().sum()) if "weather_temperature_c" in sub_c else 0
                    def _plage(col):
                        s = sub_c.loc[sub_c[col].notna(), "hour_utc"] if col in sub_c else None
                        return f"du {s.min():%d/%m/%Y} au {s.max():%d/%m/%Y %H:%M} UTC" if s is not None and len(s) else "aucune donnée"
                    
                    cross_info_html = f"""
                    <div style='background: #eef2f7; border-left: 5px solid #2980b9; padding: 16px 20px; border-radius: 8px; margin-bottom: 20px;'>
                        <h4 style='margin: 0 0 8px 0; color: #2c3e50;'>[[ico:info]] Architecture du Jumeau Numérique & Analyse Temporelle</h4>
                        <p style='margin: 0 0 10px 0; font-size: 13px; color: #34495e;'>
                            Dans notre plateforme Smart City, chaque flux possède sa propre dynamique temporelle :
                        </p>
                        <ul style='margin: 0; padding-left: 20px; font-size: 13px; color: #2c3e50; line-height: 1.6;'>
                            <li><b>[[ico:thermometer]] Météo (Open-Meteo) :</b> Modèle ECMWF IFS aux coordonnées de la ville - <b>{nb_met}</b> heures, {_plage('weather_temperature_c')}.</li>
                            <li><b>[[ico:wind]] Pollution (OpenAQ) :</b> Capteurs IoT - <b>{nb_poll}</b> heures mesurées, {_plage('pollution_no2_ugm3')}.</li>
                            <li><b>[[ico:bike]] Mobilité (CityBikes) :</b> Relevés par micro-batch - <b>{nb_mob}</b> créneau(x) horaire(s), {_plage('mobility_bikes_available')}.</li>
                        </ul>
                    </div>
                    """
                    display(HTML(cross_info_html))
                    
                    # Tableau filtré des observations actives récentes (sans NaN bruts)
                    has_active = pd.Series(False, index=sub_c.index)
                    if "pollution_no2_ugm3" in sub_c:
                        has_active = has_active | sub_c["pollution_no2_ugm3"].notna()
                    if "mobility_bikes_available" in sub_c:
                        has_active = has_active | sub_c["mobility_bikes_available"].notna()
                    
                    sub_active = sub_c[has_active].sort_values("hour_utc", ascending=False).copy()
                    
                    if not sub_active.empty:
                        display(HTML("<h4>[[ico:clock]] Dernières Observations Multi-Sources Actives (Novembre 2026)</h4>"))
                        cols_show = ["hour_utc", "city_name", "pollution_no2_ugm3", "pollution_pm25_ugm3", "pollution_pm10_ugm3", "mobility_bikes_available", "mobility_free_slots", "statut_jointure"]
                        cols_exist = [c for c in cols_show if c in sub_active.columns]
                        table_display = sub_active[cols_exist].head(15).copy().astype(object).fillna("-")
                        table_display.columns = [c.replace("pollution_", "Pollution ").replace("mobility_", "Vélos ").replace("_ugm3", " (µg/m³)").replace("hour_utc", "Date & Heure UTC") for c in table_display.columns]
                        display(table_display)
                    else:
                        display(HTML("<h4>[[ico:clock]] Dernières Observations Météorologiques Enregistrées</h4>"))
                        cols_w_show = ["hour_utc", "city_name", "weather_temperature_c", "weather_wind_speed_ms", "weather_precipitation_mm"]
                        cols_w_exist = [c for c in cols_w_show if c in sub_c.columns]
                        table_w = sub_c[cols_w_exist].sort_values("hour_utc", ascending=False).head(15).copy().astype(object).fillna("-")
                        display(table_w)
                else:
                    display(HTML(f"<p style='color:#e67e22;'>[[ico:alert]] Aucune heure croisée trouvée pour <code>{selected_id}</code>.</p>"))
            else:
                display(HTML("<p style='color:#e74c3c;'>[[ico:x-circle]] Table Gold de croisement introuvable.</p>"))
        
        # ==========================================================
        # ONGLET 5 : HISTORIQUE ET COUVERTURE DES DONNEES
        # ==========================================================
        with out_h:
            try:
                display(HTML("<h4>[[ico:thermometer]] Historique météo sur plusieurs années</h4>"))
                df_wh = lire_historique("wh", ["kpis/history/weather_daily_visibility.parquet"])
                if df_wh is None or df_wh.empty:
                    df_wh = df_w
                sub_h = df_wh[df_wh["city_id"] == selected_id].copy() if df_wh is not None else pd.DataFrame()
                if sub_h.empty:
                    display(HTML("<p style='color:#e74c3c;'>[[ico:x-circle]] Table Gold météo introuvable.</p>"))
                else:
                    sub_h["jour"] = pd.to_datetime(sub_h["day_utc"], utc=True)
                    sub_h["annee"] = sub_h["jour"].dt.year
                    sub_h["mois"] = sub_h["jour"].dt.month
                    mensuel = sub_h.groupby(["annee", "mois"]).agg(
                        temperature=("temperature_mean", "mean"), pluie=("precipitation_observed", "sum"), jours=("jour", "count")
                    ).reset_index()
                    mensuel = mensuel[mensuel["jours"] >= 25]  # les mois incomplets ne sont pas tracés
                    fig, axes = plt.subplots(1, 2, figsize=(13, 4), dpi=100)
                    for annee, g in mensuel.groupby("annee"):
                        axes[0].plot(g["mois"], g["temperature"], marker="o", ms=3, label=str(annee))
                        axes[1].plot(g["mois"], g["pluie"], marker="o", ms=3, label=str(annee))
                    axes[0].set_title(f"Température moyenne par mois - {selected_name}", fontsize=11, fontweight="bold")
                    axes[0].set_ylabel("°C")
                    axes[1].set_title("Précipitations cumulées par mois (mm)", fontsize=11, fontweight="bold")
                    for ax in axes:
                        ax.set_xticks(range(1, 13)); ax.set_xlabel("Mois"); ax.grid(True, linestyle="--", alpha=0.4)
                    axes[0].legend(ncol=2, fontsize=8, title="Année")
                    plt.tight_layout()
                    plt.show()
                    vis = (sub_h.groupby("annee")["visibility_mean"].apply(lambda s: s.notna().mean() * 100)
                           if "visibility_mean" in sub_h else pd.Series(dtype=float))
                    if not vis.empty:
                        lignes_v = "".join(f"<td style='padding:4px 10px;text-align:center;'>{int(a)}<br><b>{v:.0f} %</b></td>" for a, v in vis.items())
                        display(HTML("<div style='margin:6px 0 14px 0;font-size:13px;'><b>Part des jours avec visibilité</b> "
                                     "(donnée modélisée Open-Meteo, absente pour Lille avant le 26/06/2024)"
                                     f"<table style='border-collapse:collapse;margin-top:4px;'><tr>{lignes_v}</tr></table></div>"))

                display(HTML("<h4>[[ico:wind]] Historique de pollution</h4>"))
                df_pm = lire_historique("pm", ["kpis/pollution_monthly.parquet"])
                if df_pm is not None and not df_pm.empty and (df_pm["city_id"] == selected_id).any():
                    pm = df_pm[df_pm["city_id"] == selected_id].copy()
                    pm["mois"] = pd.to_datetime(pm["month"], utc=True)
                    fig, ax = plt.subplots(figsize=(11, 3.8), dpi=100)
                    for pol, g in pm[pm["pollutant"] != "co"].groupby("pollutant"):  # CO exclu : échelle très différente
                        g = g.sort_values("mois")
                        ax.plot(g["mois"], g["value_mean"], marker="o", ms=3, label=str(pol).upper())
                    ax.set_title(f"Concentration moyenne mensuelle - {selected_name}", fontsize=11, fontweight="bold")
                    ax.set_ylabel("µg/m³"); ax.grid(True, linestyle="--", alpha=0.4); ax.legend(ncol=5, fontsize=8)
                    plt.tight_layout(); plt.show()
                else:
                    display(HTML("<p style='color:#7f8c8d;'>[[ico:info]] Pas encore d'historique mensuel : la table <code>kpis/pollution_monthly.parquet</code> "
                                 "sera produite après l'intégration des archives OpenAQ (<code>scripts/openaq_archive.py</code>). "
                                 "Aujourd'hui la pollution ne couvre que quelques semaines.</p>"))

                # Couverture des données et état des archives
                def _plage_h(df, col):
                    if df is None or df.empty or col not in df:
                        return None
                    s = pd.to_datetime(df[col], utc=True, errors="coerce").dropna()
                    return None if s.empty else (s.min(), s.max())

                def _nb_objets(bucket, prefix):
                    if (bucket, prefix) in _cache_nb:
                        return _cache_nb[(bucket, prefix)]
                    try:
                        n_obj = sum(1 for _ in itertools.islice(client.list_objects(bucket, prefix=prefix, recursive=True), 50000))
                        if n_obj > 0:
                            _cache_nb[(bucket, prefix)] = n_obj
                        return n_obj
                    except Exception:
                        return None

                df_pa = lire_historique("pa", ["kpis/history/pollution_hourly_archive.parquet"], ["hour_utc"])
                df_mh = lire_historique("mh", ["kpis/history/mobility_hourly_archive.parquet"])
                lignes_c = ""
                for nom, df, col in (("Météo (Open-Meteo)", df_wh, "day_utc"), ("Pollution (OpenAQ)", df_pa if df_pa is not None else df_p, "hour_utc"),
                                     ("Mobilité (CityBikes)", df_mh if df_mh is not None else df_m_hourly, "hour_utc")):
                    plage = _plage_h(df, col)
                    if plage is None:
                        lignes_c += f"<tr><td style='padding:5px 10px;text-align:left;'>{nom}</td><td colspan='3' style='padding:5px 10px;color:#7f8c8d;'>Indisponible</td></tr>"
                        continue
                    jours = (plage[1] - plage[0]).total_seconds() / 86400
                    duree = f"{jours * 24:.0f} h" if jours < 2 else f"{jours:,.0f} jours"
                    note = "Historique de plusieurs années" if jours >= 365 else "Historique court : archives à intégrer"
                    couleur = "#27ae60" if jours >= 365 else "#e67e22"
                    lignes_c += (f"<tr><td style='padding:5px 10px;text-align:left;'>{nom}</td><td style='padding:5px 10px;text-align:left;'>{plage[0]:%d/%m/%Y} au {plage[1]:%d/%m/%Y}</td>"
                                 f"<td style='padding:5px 10px;text-align:left;'>{duree}</td><td style='padding:5px 10px;text-align:left;color:{couleur};font-weight:bold;'>{note}</td></tr>")
                display(HTML("<h4>[[ico:clock]] Couverture des données</h4>"
                             "<table style='border-collapse:collapse;font-size:13px;'><tr style='color:#7f8c8d;text-align:left;'>"
                             "<th style='padding:5px 10px;text-align:left;'>Source</th><th style='padding:5px 10px;text-align:left;'>Période</th>"
                             "<th style='padding:5px 10px;text-align:left;'>Durée</th><th style='padding:5px 10px;text-align:left;'>État</th></tr>"
                             f"{lignes_c}</table>"))

                bronze = os.getenv("MINIO_BUCKET_BRONZE", "urbanhub-bronze")
                silver_b = os.getenv("MINIO_BUCKET_SILVER", "urbanhub-silver")
                lignes_a = ""
                for nom, bucket, prefixe in (("CityBikes (Silver horaire 2024-2026)", silver_b, "citybikes-archive/"),
                                             ("OpenAQ (archive publique S3)", bronze, "openaq-archive/"),
                                             ("Open-Meteo avec visibilité 2022-2026", silver_b, "weather_openmeteo_archive/")):
                    n = _nb_objets(bucket, prefixe)
                    if n is None:
                        etat = "<span style='color:#7f8c8d;'>[[ico:x-circle]] Illisible</span>"
                    elif n == 0:
                        etat = "<span style='color:#e67e22;'>[[ico:alert]] Non publiée</span>"
                    else:
                        etat = f"<span style='color:#27ae60;font-weight:bold;'>[[ico:check-circle]] Publiée ({n:,} objets)</span>"
                    lignes_a += f"<tr><td style='padding:5px 10px;text-align:left;'>{nom}</td><td style='padding:5px 10px;text-align:left;'>{etat}</td></tr>"
                display(HTML("<h4>[[ico:info]] Archives sur 5 ans (2022 à 2026)</h4>"
                             "<table style='border-collapse:collapse;font-size:13px;'><tr style='color:#7f8c8d;'>"
                             "<th style='padding:5px 10px;text-align:left;'>Archive</th><th style='padding:5px 10px;text-align:left;'>État dans le Data Lake</th></tr>"
                             f"{lignes_a}</table>"
                             "<p style='color:#95a5a6;font-size:12px;'>Données de CityBikes sous licence ODbL (Bike-share data by CityBikes contributors, data.citybik.es), "
                             "d'OpenAQ et d'Open-Meteo.com (CC BY 4.0).</p>"))
            except Exception as erreur:
                display(HTML(f"<p style='color:#e74c3c;'>[[ico:x-circle]] Historique indisponible : {erreur}</p>"))

        # ==========================================================
        # ONGLET 6 : ANALYSES SUR 5 ANS (ville sélectionnée)
        # ==========================================================
        with out_a:
            try:
                SEUILS = {"pm25": 15.0, "pm10": 45.0, "no2": 25.0, "o3": 100.0}
                if selected_id not in _cache_analyses:
                    cols_h = ["city_id", "hour_utc", "weather_temperature_c", "weather_wind_speed_ms", "weather_precipitation_mm",
                              "pollution_no2_ugm3", "pollution_pm25_ugm3", "pollution_pm10_ugm3", "pollution_o3_ugm3"]
                    h_all = lire_historique("h_all", ["kpis/history/city_hourly_history.parquet"], cols_h)
                    d_all = lire_historique("d_all", ["kpis/history/weather_daily_visibility.parquet"],
                                            ["city_id", "day_utc", "temperature_min", "temperature_max", "wind_speed_max", "precipitation_observed"])
                    if h_all is None or d_all is None:
                        raise RuntimeError("tables kpis/history/ introuvables dans le Gold")
                    h_c = h_all[h_all["city_id"] == selected_id].copy()
                    h_c["hour_utc"] = pd.to_datetime(h_c["hour_utc"], utc=True)
                    d_c = d_all[d_all["city_id"] == selected_id].copy().sort_values("day_utc")
                    d_c["jour"] = pd.to_datetime(d_c["day_utc"], utc=True)
                    _cache_analyses[selected_id] = (h_c, d_c)
                ha, da = _cache_analyses[selected_id]

                # --- 1. Records et épisodes météo anormaux
                display(HTML("<h4>[[ico:thermometer]] Records et épisodes météo anormaux (2022-2026)</h4>"))
                def _carte(titre, valeur, jour_, couleur):
                    return (f"<div style='background:#f8f9fa;border-left:5px solid {couleur};padding:10px 14px;border-radius:6px;'>"
                            f"<div style='color:#7f8c8d;font-size:12px;font-weight:bold;'>{titre}</div>"
                            f"<div style='font-size:22px;font-weight:bold;color:#2c3e50;'>{valeur}</div>"
                            f"<div style='color:#95a5a6;font-size:12px;'>{jour_:%d/%m/%Y}</div></div>")
                r_chaud = da.loc[da["temperature_max"].idxmax()]; r_froid = da.loc[da["temperature_min"].idxmin()]
                r_vent = da.loc[da["wind_speed_max"].idxmax()]; r_pluie = da.loc[da["precipitation_observed"].idxmax()]
                display(HTML("<div style='display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin-bottom:12px;'>"
                             + _carte("JOUR LE PLUS CHAUD", f"{r_chaud['temperature_max']:.1f} °C", r_chaud["jour"], "#e74c3c")
                             + _carte("JOUR LE PLUS FROID", f"{r_froid['temperature_min']:.1f} °C", r_froid["jour"], "#3498db")
                             + _carte("VENT MAXIMAL", f"{r_vent['wind_speed_max']:.1f} m/s", r_vent["jour"], "#f39c12")
                             + _carte("JOUR LE PLUS PLUVIEUX", f"{r_pluie['precipitation_observed']:.1f} mm", r_pluie["jour"], "#2980b9")
                             + "</div>"))
                d2 = da.copy()
                d2["saison"] = d2["jour"].dt.month.map(lambda m: "hiver" if m in (12, 1, 2) else "printemps" if m in (3, 4, 5) else "été" if m in (6, 7, 8) else "automne")
                d2["chaud"] = d2["temperature_max"] > d2.groupby("saison")["temperature_max"].transform(lambda s: s.quantile(0.95))
                d2["froid"] = d2["temperature_min"] < d2.groupby("saison")["temperature_min"].transform(lambda s: s.quantile(0.05))
                eps = []
                for flag, nom, col, fn in (("chaud", "Chaud", "temperature_max", "max"), ("froid", "Froid", "temperature_min", "min")):
                    run = (d2[flag] != d2[flag].shift()).cumsum()
                    for _, bloc in d2[d2[flag]].groupby(run[d2[flag]]):
                        if len(bloc) >= 3:
                            eps.append({"Type": nom, "Début": f"{bloc['jour'].min():%d/%m/%Y}", "Fin": f"{bloc['jour'].max():%d/%m/%Y}",
                                        "Jours": len(bloc), "Extrême (°C)": round(bloc[col].max() if fn == "max" else bloc[col].min(), 1)})
                if eps:
                    display(HTML(f"<p style='font-size:13px;color:#34495e;'>{len(eps)} épisodes d'au moins 3 jours au-dessus du P95 (chaud) ou sous le P05 (froid) de la saison. Les plus longs :</p>"))
                    display(pd.DataFrame(eps).sort_values(["Jours", "Extrême (°C)"], ascending=[False, False]).head(8).reset_index(drop=True))

                # --- 2. Seuils OMS par année
                display(HTML("<h4>[[ico:wind]] Dépassements des lignes directrices OMS (moyenne sur 24 h)</h4>"))
                hs = ha.set_index("hour_utc")
                lignes_o = {}
                for pol, seuil in SEUILS.items():
                    c = f"pollution_{pol}_ugm3"
                    if c in hs:
                        q = hs[c].resample("D").mean().dropna()
                        if len(q):
                            nb_j = q.groupby(q.index.year).size()
                            part = (q > seuil).groupby(q.index.year).mean().mul(100).round(1)
                            lignes_o[pol.upper()] = part[nb_j >= 30]  # années avec moins de 30 jours de mesure ignorées
                if lignes_o:
                    oms = pd.DataFrame(lignes_o); oms.index.name = "Année"
                    display(HTML("<p style='font-size:13px;color:#34495e;'>Part des jours au-dessus du seuil (%), seuils : "
                                 + ", ".join(f"{k.upper()} {v:g} µg/m³" for k, v in SEUILS.items()) + ".</p>"))
                    display(oms.astype(object).fillna("-"))

                # --- 3. Profil horaire et ratio au seuil OMS
                fig, axes = plt.subplots(1, 2, figsize=(13, 4), dpi=100)
                loc = ha["hour_utc"].dt.tz_convert("Europe/Paris").dt.hour
                ratios = {}
                for pol in SEUILS:
                    c = f"pollution_{pol}_ugm3"
                    if c in ha and ha[c].notna().any():
                        axes[0].plot(ha.groupby(loc)[c].mean(), marker="o", ms=3, label=pol.upper())
                        ratios[pol.upper()] = ha[c].mean() / SEUILS[pol]
                axes[0].set_title(f"Profil horaire moyen (heure locale) - {selected_name}", fontsize=11, fontweight="bold")
                axes[0].set_xticks(range(0, 24, 2)); axes[0].set_xlabel("Heure"); axes[0].set_ylabel("µg/m³"); axes[0].legend(ncol=4, fontsize=8); axes[0].grid(True, linestyle="--", alpha=0.4)
                axes[1].bar(list(ratios), list(ratios.values()), color=["#c0392b" if v >= 1 else "#27ae60" for v in ratios.values()])
                axes[1].axhline(1, color="red", linestyle="--", linewidth=1)
                axes[1].set_title("Concentration moyenne / seuil OMS (1 = seuil)", fontsize=11, fontweight="bold"); axes[1].grid(True, axis="y", linestyle="--", alpha=0.4)
                plt.tight_layout(); plt.show()

                # --- 4. Météo et pollution
                display(HTML("<h4>[[ico:info]] Influence de la météo sur la pollution</h4>"))
                fig, axes = plt.subplots(1, 2, figsize=(13, 4), dpi=100)
                bins = [0, 2, 4, 6, 8, 12, 40]; labels_b = ["0-2", "2-4", "4-6", "6-8", "8-12", ">12"]
                lectures = []
                for pol, couleur in (("no2", "#d9534f"), ("pm25", "#f0ad4e")):
                    c = f"pollution_{pol}_ugm3"
                    s = ha[["weather_wind_speed_ms", c]].dropna()
                    if len(s) > 100:
                        g = s.assign(cl=pd.cut(s["weather_wind_speed_ms"], bins, labels=labels_b)).groupby("cl", observed=True)[c].agg(["mean", "sem"])
                        axes[0].errorbar(range(len(g)), g["mean"], yerr=g["sem"], marker="o", capsize=3, label=pol.upper(), color=couleur)
                        axes[0].set_xticks(range(len(g))); axes[0].set_xticklabels(g.index)
                        lectures.append(f"{pol.upper()} et vent : rho = {s['weather_wind_speed_ms'].rank().corr(s[c].rank()):+.2f} sur {len(s):,} heures")
                axes[0].set_title("Dispersion par le vent", fontsize=11, fontweight="bold"); axes[0].set_xlabel("Vitesse du vent (m/s)"); axes[0].set_ylabel("µg/m³"); axes[0].legend(); axes[0].grid(True, linestyle="--", alpha=0.4)
                sp = ha[["weather_precipitation_mm", "pollution_pm10_ugm3"]].dropna()
                if len(sp) > 100:
                    sans, avec = sp.loc[sp["weather_precipitation_mm"] <= 0, "pollution_pm10_ugm3"], sp.loc[sp["weather_precipitation_mm"] > 0, "pollution_pm10_ugm3"]
                    axes[1].boxplot([sans, avec], tick_labels=[f"sans pluie\n(n={len(sans):,})", f"pluie\n(n={len(avec):,})"], showfliers=False)
                    lectures.append(f"PM10 : médiane {avec.median():.1f} µg/m³ avec pluie contre {sans.median():.1f} sans")
                axes[1].set_title("PM10 avec et sans pluie", fontsize=11, fontweight="bold"); axes[1].set_ylabel("µg/m³"); axes[1].grid(True, axis="y", linestyle="--", alpha=0.4)
                plt.tight_layout(); plt.show()
                if lectures:
                    display(HTML("<ul style='font-size:13px;color:#2c3e50;'>" + "".join(f"<li>{x}</li>" for x in lectures) + "</ul>"))
                # --- 5. Mobilité (archive CityBikes) et météo
                display(HTML("<h4>[[ico:bike]] Mobilité en vélos en libre-service (nov. 2024 à sept. 2026)</h4>"))
                mh_all = lire_historique("mh", ["kpis/history/mobility_hourly_archive.parquet"])
                mh = mh_all[mh_all["city_id"] == selected_id].copy() if mh_all is not None else pd.DataFrame()
                if mh.empty:
                    display(HTML("<p style='color:#7f8c8d;'>[[ico:info]] Historique de mobilité non publié : la table "
                                 "<code>kpis/history/mobility_hourly_archive.parquet</code> est produite par <code>scripts/citybikes_archive_gold.py</code>.</p>"))
                else:
                    mh["hour_utc"] = pd.to_datetime(mh["hour_utc"], utc=True)
                    mh["usage"] = mh["movements"] / mh["stations"]
                    loc_m = mh["hour_utc"].dt.tz_convert("Europe/Paris")
                    mh["h_loc"] = loc_m.dt.hour
                    mh["we"] = loc_m.dt.day_of_week >= 5
                    jm = mh.merge(ha[["hour_utc", "weather_precipitation_mm", "weather_temperature_c"]], on="hour_utc", how="inner")
                    jm = jm[jm["usage"].notna() & jm["h_loc"].between(7, 19)]  # heures de jour : évite que la température ne reflète que l'heure
                    fig, axes = plt.subplots(1, 3, figsize=(16, 4), dpi=100)
                    for we, lab, couleur in ((False, "jours ouvrés", "#2980b9"), (True, "week-end", "#e67e22")):
                        axes[0].plot(mh[mh["we"] == we].groupby("h_loc")["usage"].mean(), marker="o", ms=3, label=lab, color=couleur)
                    axes[0].set_title("Usage moyen par heure (vélos déplacés par station)", fontsize=11, fontweight="bold")
                    axes[0].set_xticks(range(0, 24, 2)); axes[0].set_xlabel("Heure locale"); axes[0].legend(); axes[0].grid(True, linestyle="--", alpha=0.4)
                    axes[1].plot(mh.groupby("h_loc")["fill_rate"].mean() * 100, marker="o", ms=3, color="#27ae60")
                    axes[1].set_title("Taux de remplissage moyen des stations (%)", fontsize=11, fontweight="bold")
                    axes[1].set_xticks(range(0, 24, 2)); axes[1].set_xlabel("Heure locale"); axes[1].grid(True, linestyle="--", alpha=0.4)
                    bornes = [-20, 0, 5, 10, 15, 20, 25, 50]
                    etiquettes = ["< 0", "0-5", "5-10", "10-15", "15-20", "20-25", "> 25"]
                    par_temp = jm.groupby(pd.cut(jm["weather_temperature_c"], bornes, labels=etiquettes), observed=True)["usage"].mean()
                    axes[2].bar(par_temp.index.astype(str), par_temp.values, color="#8e44ad")
                    axes[2].set_title("Usage selon la température (7 h à 19 h)", fontsize=11, fontweight="bold")
                    axes[2].set_xlabel("Température (°C)"); axes[2].grid(True, axis="y", linestyle="--", alpha=0.4)
                    plt.tight_layout(); plt.show()
                    sans_p = jm.loc[jm["weather_precipitation_mm"] <= 0, "usage"]
                    avec_p = jm.loc[jm["weather_precipitation_mm"] > 0, "usage"]
                    lignes_m = []
                    if len(sans_p) > 100 and len(avec_p) > 100:
                        lignes_m.append(f"Entre 7 h et 19 h, l'usage moyen est de {avec_p.mean():.2f} avec pluie contre {sans_p.mean():.2f} sans "
                                        f"({(avec_p.mean() / sans_p.mean() - 1) * 100:+.0f} %), sur {len(avec_p):,} et {len(sans_p):,} heures.")
                    if len(par_temp) > 1:
                        lignes_m.append(f"Tranche de température où l'usage est le plus élevé : {par_temp.idxmax()} °C.")
                    lignes_m.append(f"Taux de remplissage moyen : {mh['fill_rate'].mean() * 100:.0f} %, en moyenne {mh['empty_stations'].mean():.0f} stations vides "
                                    f"et {mh['full_stations'].mean():.0f} saturées sur {mh['stations'].max():,.0f}.")
                    display(HTML("<ul style='font-size:13px;color:#2c3e50;'>" + "".join(f"<li>{x}</li>" for x in lignes_m) + "</ul>"
                                 "<p style='color:#95a5a6;font-size:12px;'>Usage = variation absolue du nombre de vélos d'une station entre deux heures (mesure plancher, "
                                 "rééquilibrage de l'opérateur inclus). L'archive CityBikes n'a pas les mois d'avril à juillet 2026 pour Paris.</p>"))
                couv = ha["pollution_no2_ugm3"].notna().mean() * 100
                display(HTML(f"<p style='color:#95a5a6;font-size:12px;'>Couverture des mesures de pollution pour {selected_name} : {couv:.0f} % des heures (NO2). "
                             "Les statistiques ne portent que sur les heures mesurées. Calculs : notebook <code>07_analyses_historiques</code>.</p>"))
            except Exception as erreur:
                display(HTML(f"<p style='color:#e74c3c;'>[[ico:x-circle]] Analyses indisponibles : {erreur}</p>"))


refresh_btn.on_click(render_dashboard)
city_selector.observe(render_dashboard, names='value')
render_dashboard()